In [1]:
# Family Relationship Knowledge Base

male = ["rajan", "kumar", "arun", "selvam", "vijay"]
female = ["lakshmi", "meena", "sita", "divya"]

parent = [
    ("rajan", "kumar"),
    ("lakshmi", "kumar"),
    ("rajan", "meena"),
    ("lakshmi", "meena"),
    ("kumar", "arun"),
    ("sita", "arun"),
    ("kumar", "divya"),
    ("sita", "divya"),
    ("meena", "vijay"),
    ("selvam", "vijay")
]


# Father
def father(F, C):
    return (F, C) in parent and F in male


# Mother
def mother(M, C):
    return (M, C) in parent and M in female


# Grandparent
def grandparent(G, C):
    for P in [x for x, y in parent if y == C]:
        if (G, P) in parent:
            return True
    return False


# Sibling
def sibling(X, Y):
    mothers_X = [m for m, c in parent if c == X and m in female]
    mothers_Y = [m for m, c in parent if c == Y and m in female]
    return X != Y and any(m in mothers_Y for m in mothers_X)


# Uncle
def uncle(U, N):
    if U not in male:
        return False

    for P in [x for x, y in parent if y == N]:
        if sibling(U, P):
            return True
    return False


# Aunt
def aunt(A, N):
    if A not in female:
        return False

    for P in [x for x, y in parent if y == N]:
        if sibling(A, P):
            return True
    return False


# Ancestor
def ancestor(A, D):
    for P, C in parent:
        if P == A:
            if C == D or ancestor(C, D):
                return True
    return False


# ---------------- QUERIES ----------------

# ?- father(F, arun).
print("Father of arun:")
for F in male:
    if father(F, "arun"):
        print("F =", F)

# ?- mother(M, vijay).
print("\nMother of vijay:")
for M in female:
    if mother(M, "vijay"):
        print("M =", M)

# ?- grandparent(G, divya).
print("\nGrandparents of divya:")
for G in male + female:
    if grandparent(G, "divya"):
        print("G =", G)

# ?- sibling(arun, S).
print("\nSiblings of arun:")
for S in male + female:
    if sibling("arun", S):
        print("S =", S)

# ?- uncle(U, vijay).
print("\nUncle of vijay:")
for U in male:
    if uncle(U, "vijay"):
        print("U =", U)

# ?- aunt(A, arun).
print("\nAunt of arun:")
for A in female:
    if aunt(A, "arun"):
        print("A =", A)

# ?- ancestor(rajan, D).
print("\nDescendants of rajan:")
for D in male + female:
    if ancestor("rajan", D):
        print("D =", D)

Father of arun:
F = kumar

Mother of vijay:
M = meena

Grandparents of divya:
G = rajan
G = lakshmi

Siblings of arun:
S = divya

Uncle of vijay:
U = kumar

Aunt of arun:
A = meena

Descendants of rajan:
D = kumar
D = arun
D = vijay
D = meena
D = divya


In [4]:
# Patient Diet and Health Advice Knowledge Base

patient = {
    "ravi": "diabetes",
    "meena": "hypertension",
    "kumar": "anaemia",
    "divya": "obesity",
    "arun": "diabetes"
}

diet = {
    "diabetes": "whole grains, oats, green vegetables, pulses, nuts",
    "hypertension": "fruits, vegetables, low-fat dairy, banana, low-salt meals",
    "anaemia": "spinach, beetroot, dates, jaggery, pulses, citrus fruits",
    "obesity": "salads, sprouts, fibre-rich foods, plenty of water, fruits"
}

avoid = {
    "diabetes": "sweets, sugary drinks, white rice in excess",
    "hypertension": "pickles, papad, processed food, excess salt",
    "anaemia": "tea or coffee immediately after meals",
    "obesity": "fried food, fast food, sugary beverages"
}


# suggest(P, Diet)
def suggest(P):
    disease = patient.get(P)
    if disease:
        return diet[disease]


# advice(P, Eat, Avoid)
def advice(P):
    disease = patient.get(P)
    if disease:
        return diet[disease], avoid[disease]


# with_disease(D, P)
def with_disease(D):
    return [P for P in patient if patient[P] == D]


# ---------------- QUERIES ----------------

# ?- suggest(ravi, D).
print("Query: suggest(ravi, D).")
print("D =", suggest("ravi"))

# ?- advice(meena, E, A).
print("\nQuery: advice(meena, E, A).")
E, A = advice("meena")
print("E =", E)
print("A =", A)

# ?- with_disease(diabetes, P).
print("\nQuery: with_disease(diabetes, P).")
for P in with_disease("diabetes"):
    print("P =", P)

# ?- suggest(divya, D).
print("\nQuery: suggest(divya, D).")
print("D =", suggest("divya"))

Query: suggest(ravi, D).
D = whole grains, oats, green vegetables, pulses, nuts

Query: advice(meena, E, A).
E = fruits, vegetables, low-fat dairy, banana, low-salt meals
A = pickles, papad, processed food, excess salt

Query: with_disease(diabetes, P).
P = ravi
P = arun

Query: suggest(divya, D).
D = salads, sprouts, fibre-rich foods, plenty of water, fruits


In [5]:
# Monkey and Banana Planning Problem

positions = ["atdoor", "atwindow", "middle"]

def get_moves(state):
    monkey, box, box_pos, has = state
    moves = []

    # Grasp
    if state == ("middle", "onbox", "middle", "hasnot"):
        moves.append(("grasp", ("middle", "onbox", "middle", "has")))

    # Climb
    if box == "onfloor" and monkey == box_pos:
        moves.append(("climb", (monkey, "onbox", box_pos, has)))

    # Push
    if box == "onfloor" and monkey == box_pos:
        for p2 in positions:
            if monkey != p2:
                moves.append((
                    f"push({monkey},{p2})",
                    (p2, "onfloor", p2, has)
                ))

    # Walk
    if box == "onfloor":
        for p2 in positions:
            if monkey != p2:
                moves.append((
                    f"walk({monkey},{p2})",
                    (p2, "onfloor", box_pos, has)
                ))

    return moves


def plan(state, visited=None):
    if visited is None:
        visited = set()

    # Goal state
    if state[3] == "has":
        return []

    if state in visited:
        return None

    visited.add(state)

    for action, new_state in get_moves(state):
        result = plan(new_state, visited.copy())

        if result is not None:
            return [action] + result

    return None


def solve(start):
    return plan(start)


# Query 1
start1 = ("atdoor", "onfloor", "atwindow", "hasnot")
print("Query 1:")
print("Plan =", solve(start1))

# Query 2
start2 = ("middle", "onfloor", "middle", "hasnot")
print("\nQuery 2:")
print("Plan =", solve(start2))

Query 1:
Plan = ['walk(atdoor,atwindow)', 'push(atwindow,atdoor)', 'push(atdoor,middle)', 'climb', 'grasp']

Query 2:
Plan = ['climb', 'grasp']


In [6]:
# Fruit Color Knowledge Base

fruit_color = {
    "apple": "red",
    "banana": "yellow",
    "cherry": "red",
    "grapes": "green",
    "guava": "green",
    "lemon": "yellow",
    "orange": "orange",
    "strawberry": "red"
}


# Query: fruit_color(F, red)
print("Query: fruit_color(F, red).")
for fruit, color in fruit_color.items():
    if color == "red":
        print("F =", fruit)


# Query: fruit_color(banana, C)
print("\nQuery: fruit_color(banana, C).")
print("C =", fruit_color["banana"])


# Query: same_colour(F1, F2)
print("\nQuery: same_colour(F1, F2).")
fruits = list(fruit_color.keys())

for i in range(len(fruits)):
    for j in range(i + 1, len(fruits)):
        f1 = fruits[i]
        f2 = fruits[j]

        if fruit_color[f1] == fruit_color[f2]:
            print("F1 =", f1, ", F2 =", f2)


# Query: all_of_colour(green, L)
print("\nQuery: all_of_colour(green, L).")
green_fruits = [f for f, c in fruit_color.items() if c == "green"]
print("L =", green_fruits)


# Query: show_all
print("\nQuery: show_all.")
for fruit, color in fruit_color.items():
    print(f"{fruit} is {color}")

print("true.")

Query: fruit_color(F, red).
F = apple
F = cherry
F = strawberry

Query: fruit_color(banana, C).
C = yellow

Query: same_colour(F1, F2).
F1 = apple , F2 = cherry
F1 = apple , F2 = strawberry
F1 = banana , F2 = lemon
F1 = cherry , F2 = strawberry
F1 = grapes , F2 = guava

Query: all_of_colour(green, L).
L = ['grapes', 'guava']

Query: show_all.
apple is red
banana is yellow
cherry is red
grapes is green
guava is green
lemon is yellow
orange is orange
strawberry is red
true.


In [7]:
# Best-First Search

# Edges
edges = {
    "a": ["b", "c"],
    "b": ["d", "e"],
    "c": ["f"],
    "e": ["g"],
    "f": ["g"]
}

# Heuristic values
h = {
    "a": 7,
    "b": 5,
    "c": 4,
    "d": 6,
    "e": 2,
    "f": 3,
    "g": 0
}


def best_first(start, goal):
    # Queue contains paths
    queue = [[start]]

    while queue:
        # Select path with lowest heuristic value
        queue.sort(key=lambda path: h[path[-1]])
        path = queue.pop(0)

        node = path[-1]

        # Goal reached
        if node == goal:
            return path

        # Generate children
        for child in edges.get(node, []):
            if child not in path:
                new_path = path + [child]
                queue.append(new_path)

    return None


# Query 1
print("Query: best_first(a, g, Path).")
path = best_first("a", "g")
print("Path =", path)


# Query 2
print("\nQuery: best_first(a, d, Path).")
path = best_first("a", "d")
print("Path =", path)

Query: best_first(a, g, Path).
Path = ['a', 'c', 'f', 'g']

Query: best_first(a, d, Path).
Path = ['a', 'b', 'd']


In [8]:
# Medical Diagnosis Expert System

# Symptoms of patients
symptoms = {
    "ravi": ["fever", "cough", "body_pain"],
    "meena": ["sneezing", "runny_nose", "sore_throat"],
    "kumar": ["fever", "chills", "sweating", "headache"]
}

# Diseases and required symptoms
disease = {
    "flu": ["fever", "cough", "body_pain"],
    "common_cold": ["sneezing", "runny_nose", "sore_throat"],
    "malaria": ["fever", "chills", "sweating"],
    "typhoid": ["fever", "headache", "stomach_pain"]
}

# Medical advice
advice = {
    "flu": "rest, fluids and consult a doctor if fever persists",
    "common_cold": "rest, warm fluids and steam inhalation",
    "malaria": "consult a doctor immediately for a blood test",
    "typhoid": "consult a doctor immediately for a Widal test"
}


# Check whether patient has all required symptoms
def has_all(patient, required_symptoms):
    return all(symptom in symptoms.get(patient, [])
               for symptom in required_symptoms)


# Diagnose patient
def diagnose(patient):
    result = []

    for d, required_symptoms in disease.items():
        if has_all(patient, required_symptoms):
            result.append(d)

    return result


# Report
def report(patient):
    diagnoses = diagnose(patient)

    for d in diagnoses:
        print(f"{patient} may have {d}. Advice: {advice[d]}")


# ---------------- QUERIES ----------------

# ?- diagnose(ravi, D).
print("Query: diagnose(ravi, D).")
for d in diagnose("ravi"):
    print("D =", d)

# ?- diagnose(meena, D).
print("\nQuery: diagnose(meena, D).")
for d in diagnose("meena"):
    print("D =", d)

# ?- diagnose(P, malaria).
print("\nQuery: diagnose(P, malaria).")
for patient in symptoms:
    if "malaria" in diagnose(patient):
        print("P =", patient)

# ?- report(kumar).
print("\nQuery: report(kumar).")
report("kumar")
print("true.")

Query: diagnose(ravi, D).
D = flu

Query: diagnose(meena, D).
D = common_cold

Query: diagnose(P, malaria).
P = kumar

Query: report(kumar).
kumar may have malaria. Advice: consult a doctor immediately for a blood test
true.


In [9]:
# Forward Chaining Expert System

# Initial facts
facts = [
    "has_hair",
    "eats_meat",
    "tawny_colour",
    "dark_spots"
]

# Rules
rules = [
    ("r1", ["has_hair"], "mammal"),
    ("r2", ["gives_milk"], "mammal"),
    ("r3", ["mammal", "eats_meat"], "carnivore"),
    ("r4", ["carnivore", "tawny_colour", "dark_spots"], "cheetah"),
    ("r5", ["carnivore", "tawny_colour", "black_stripes"], "tiger")
]


# Forward chaining
def forward():
    while True:
        fired = False

        for rule_id, conditions, conclusion in rules:

            # Check if conclusion is not already a fact
            # and all conditions are facts
            if conclusion not in facts and all(c in facts for c in conditions):

                facts.append(conclusion)

                print(f"Rule {rule_id} fired: derived {conclusion}")

                fired = True
                break

        if not fired:
            break


# ---------------- QUERY 1 ----------------
print("Query: forward().")
forward()
print("true.")


# ---------------- QUERY 2 ----------------
print("\nQuery: fact(cheetah).")

if "cheetah" in facts:
    print("true.")
else:
    print("false.")


# ---------------- QUERY 3 ----------------
print("\nQuery: fact(tiger).")

if "tiger" in facts:
    print("true.")
else:
    print("false.")


# ---------------- QUERY 4 ----------------
print("\nQuery: findall(F, fact(F), L).")
print("L =", facts)

Query: forward().
Rule r1 fired: derived mammal
Rule r3 fired: derived carnivore
Rule r4 fired: derived cheetah
true.

Query: fact(cheetah).
true.

Query: fact(tiger).
false.

Query: findall(F, fact(F), L).
L = ['has_hair', 'eats_meat', 'tawny_colour', 'dark_spots', 'mammal', 'carnivore', 'cheetah']


In [10]:
# Backward Chaining Expert System

# Facts
facts = [
    "has_hair",
    "eats_meat",
    "tawny_colour",
    "black_stripes"
]

# Rules
rules = [
    ("r1", ["has_hair"], "mammal"),
    ("r2", ["gives_milk"], "mammal"),
    ("r3", ["mammal", "eats_meat"], "carnivore"),
    ("r4", ["carnivore", "tawny_colour", "dark_spots"], "cheetah"),
    ("r5", ["carnivore", "tawny_colour", "black_stripes"], "tiger")
]


# Backward chaining
def bc(goal):
    # Goal is already a fact
    if goal in facts:
        return True

    # Try rules whose conclusion matches the goal
    for rule_id, conditions, conclusion in rules:
        if conclusion == goal:
            if bc_all(conditions):
                return True

    return False


def bc_all(conditions):
    for goal in conditions:
        if not bc(goal):
            return False
    return True


# Version that prints the reasoning
def prove(goal, depth):
    print(" " * depth + f"Proving {goal}")

    # Known fact
    if goal in facts:
        print(" " * depth + f"{goal} is a known fact")
        return True

    # Find a rule for the goal
    for rule_id, conditions, conclusion in rules:
        if conclusion == goal:
            print(" " * depth + f"using rule {rule_id}")

            new_depth = depth + 2

            for condition in conditions:
                if not prove(condition, new_depth):
                    return False

            return True

    return False


# ---------------- QUERIES ----------------

# ?- bc(tiger).
print("Query: bc(tiger).")
print("true." if bc("tiger") else "false.")


# ?- bc(cheetah).
print("\nQuery: bc(cheetah).")
print("true." if bc("cheetah") else "false.")


# ?- bc(mammal).
print("\nQuery: bc(mammal).")
print("true." if bc("mammal") else "false.")


# ?- prove(tiger, 0).
print("\nQuery: prove(tiger, 0).")
if prove("tiger", 0):
    print("true.")
else:
    print("false.")

Query: bc(tiger).
true.

Query: bc(cheetah).
false.

Query: bc(mammal).
true.

Query: prove(tiger, 0).
Proving tiger
using rule r5
  Proving carnivore
  using rule r3
    Proving mammal
    using rule r1
      Proving has_hair
      has_hair is a known fact
    Proving eats_meat
    eats_meat is a known fact
  Proving tawny_colour
  tawny_colour is a known fact
  Proving black_stripes
  black_stripes is a known fact
true.
